In [1]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(bnlearn)')

In [2]:
#set up for experimentation
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_HC"
output_dir.mkdir(parents=True,exist_ok=True)

In [7]:
import time
#run hc function
def run_hc(df, seed=1):
    #remove NA rows
    df_clean = df.dropna().copy()

    start = time.time()
    with (ro.default_converter+pandas2ri.converter).context():
        ro.globalenv["df"]=conversion.py2rpy(df_clean)
        ro.globalenv["seed"]= seed

        ro.r('''
        library(bnlearn)
        data_df <- as.data.frame(df)
        
        any_disc <- FALSE
        any_cont <- FALSE

        for (i in seq_along(data_df)) {
          col <- data_df[[i]]
          if (is.numeric(col)) {
            vals <- unique(col[!is.na(col)])
            if (length(vals) <= 6 && all(abs(vals - round(vals)) < 1e-8)) {
              data_df[[i]] <- factor(col)
              any_disc <- TRUE
            } else {
              data_df[[i]] <- as.numeric(col)
              any_cont <- TRUE
            }
          } else {
            data_df[[i]] <- factor(col)
            any_disc <- TRUE
          }
        }

        #detect if there are mixed types, continuous only, or discrete only
        if (any_disc && any_cont) {
          chosen_score <- "bic-cg"    # mixed conditional Gaussian [web:148][web:154]
        } else if (any_disc && !any_cont) {
          chosen_score <- "bde"       # discrete-only
        } else if (!any_disc && any_cont) {
          chosen_score <- "bic-g"     # Gaussian-only
        } else {
          stop("No usable variables (neither discrete nor continuous).")
        }
        
        set.seed(seed)

        #run hill climbing with conditional score
        dag_hc <- hc(data_df, score=chosen_score)

        adj <- amat(dag_hc)
        nodes <- colnames(adj)
        ''')
        end = time.time()
        
        #convert adjacency matrix in R back to python
        adj = conversion.rpy2py(ro.r("adj"))
        nodes=list(ro.r("nodes"))

    #coerce adjacency matrix into numeric 2D array
    print(f"HC took {(end - start)/60:.2f} minutes")
    adj=np.asarray(adj, dtype=int)
    return adj,nodes

In [3]:
import networkx as nx
import matplotlib.pyplot as plt

#graph drawing functions
def draw_graph(adj, nodes, out_path):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i,j]==1:
                G.add_edge(src,tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [4]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [8]:
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)

# Run HC on this dataset
adj,nodes = run_hc(df, seed=1)

out_path = output_dir/"NIJ_graph_HC"
draw_graphviz_dag(adj, out_path, nodes)

HC took 0.02 minutes


In [9]:
G = nx.DiGraph()
#add nodes
G.add_nodes_from(nodes)

#add directed edges, adj[i,j] = 1 => i -> j
for i, src in enumerate(nodes):
    for j, tgt in enumerate(nodes):
        if adj[i, j] == 1:
            G.add_edge(src, tgt)

In [12]:
import networkx as nx
from collections import Counter

AGE_PREFIX = "Age_at_Release_"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))

print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mb.edges()))


Original nodes: 21
After age-merge: 15
Markov blanket nodes: 4
Original edges: 76
After age-merge: 42
Markov blanket edges: 4


In [11]:
nodes = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
adj_df = pd.DataFrame(A, index=nodes, columns=nodes)
adj_df.to_csv(output_dir/ "NIJ_graph_HC_adj.csv", index=True)

In [13]:
node_labels = list(G_mb.nodes())          
adj = nx.to_numpy_array(G_mb, nodelist=node_labels, dtype=int)

out_path = output_dir/"NIJ_graph_HC_PRUNED"
draw_graphviz_dag(adj, out_path, node_labels=node_labels, engine="dot")